# LC 42 — Trapping Rain Water
**Day-63 | Hard | Two Pointers**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Water above any bar equals
<code>min(left_max, right_max) - height[i]</code>.
Two pointers let us compute this in one pass by always processing
the side with the smaller maximum — because that side's water level
is already fully constrained.
</div>

## Official Problem Statement

Given `n` non-negative integers representing an elevation map where
the width of each bar is 1, compute how much water it can trap
after raining.

**Example 1:**
```
Input:  height = [0,1,0,2,1,0,1,3,2,1,2,1]
Output: 6
```
**Example 2:**
```
Input:  height = [4,2,0,3,2,5]
Output: 9
```

**Constraints:**
- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

## What This Is Actually Asking

Think of the array as a cross-section of a terrain.
Rain fills every "valley" between two taller walls.
For each position, water can only be held up to the height
of the shorter of the two tallest walls on either side.
We need the total units of water across all positions.
The challenge is doing this efficiently without pre-computing
prefix/suffix max arrays for every index.

## Walk Through an Example by Hand

`height = [4, 2, 0, 3, 2, 5]`

```
left=0  right=5  left_max=0  right_max=0  water=0

Step 1: h[0]=4, h[5]=5
  left_max(0) < right_max(0)? No (equal), go right side.
  right_max = max(0,5)=5. water += 5-5=0. right=4.

Step 2: h[0]=4, h[4]=2
  left_max(0) < right_max(5)? Yes.
  left_max = max(0,4)=4. water += 4-4=0. left=1.

Step 3: h[1]=2, h[4]=2
  left_max(4) < right_max(5)? Yes.
  left_max stays 4. water += 4-2=2. left=2. water=2.

Step 4: h[2]=0, h[4]=2
  left_max(4) < right_max(5)? Yes.
  water += 4-0=4. left=3. water=6.

Step 5: h[3]=3, h[4]=2
  left_max(4) < right_max(5)? Yes.
  water += 4-3=1. left=4. water=7. Hmm...
```
Wait, expected=9. Let's re-check: left_max updates BEFORE
adding water. At step 3, left_max becomes max(4,2)=4 first,
then water += 4-2=2. Correct total = 9 when all steps done.

## The Picture

```
height = [0,1,0,2,1,0,1,3,2,1,2,1]  answer = 6

         #               <- wall (height=3)
    #    #  #  #         <- walls
 #  # #  # ## ##  #      <- raw bars

Trapped water (~):

         |
    |~~~~|  |  |
 |  |~|~~|~||~||  |
 0  1  2  3  4  5  6  7  8  9 10 11

Two-pointer state:

 L --->               <--- R
 |left_max tracks      right_max tracks|
 |tallest seen left     tallest seen right|

 Key rule: process the SMALLER side.
 If left_max < right_max:
   water at L = left_max - height[L]  (right wall is taller,
                                        so it WILL hold water)
   advance L
 else:
   water at R = right_max - height[R]
   advance R
```

## When To Use This Pattern

- When computing a value **at each index** depends on something
  to its left AND right, think two pointers from both ends.
- When you can make a **greedy choice** based on which side
  is "safer" (smaller max), think two-pointer with tracked max.
- When a brute-force needs prefix/suffix arrays (O(n) space),
  think whether two pointers eliminate that extra space.
- When constraints are `O(n)` time and `O(1)` space, think
  two pointers over dynamic programming.
- When the problem is about water, containers, or heights
  with walls on two sides, think this exact pattern.

## The Approach

Place one pointer at each end of the array and track the maximum
height seen so far from each side.
At every step, advance the pointer on the side with the smaller
running maximum — because water at that position is fully
determined by the smaller side's max (the taller side is
guaranteed to hold it).
Add the difference between the running max and the current
height to the running water total (clamp at zero).
Stop when the two pointers meet.

In [2]:
from typing import List

In [3]:
def test_harness(func):
    cases = [
        # (input, expected, label)
        ([0,1,0,2,1,0,1,3,2,1,2,1],  6, "classic"),
        ([4,2,0,3,2,5],              9, "simple 6"),
        ([3,0,3],                    3, "valley"),
        ([1],                        0, "single"),
        ([1,2],                      0, "two bars"),
        ([0,0,0],                    0, "all zeros"),
        ([5,4,3,2,1],                0, "descending"),
        ([1,2,3,4,5],                0, "ascending"),
        ([2,0,2],                    2, "symmetric"),
    ]
    passed = 0
    for height, expected, label in cases:
        result = func(height)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} [{label}]: "
                f"got {result}, expected {expected}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [5]:
def trap(height: List[int]) -> int:
    """
    Trap rainwater using two pointers.

    Strategy:
        - left=0, right=n-1, left_max=0, right_max=0, water=0
        - While left < right:
            - If left_max < right_max: process left side
                left_max = max(left_max, height[left])
                water += left_max - height[left]
                left++
            - Else: process right side
                right_max = max(right_max, height[right])
                water += right_max - height[right]
                right--

    Args:
        height: List of non-negative integers (elevation map)

    Returns:
        Total units of trapped water

    Time:  O(n) — single pass
    Space: O(1) — only pointers and counters
    """
    # Debug: print input
    print(f"[debug] height={height}")
    if not height: return 0
    l, r = 0, len(height) -1
    res = 0
    leftMax, rightMax = height[l], height[r]
    while l<r :
        if leftMax <= rightMax:
            l += 1
            leftMax = max(leftMax, height[l])
            res += leftMax - height[l]
        else:
            r -= 1
            rightMax = max(rightMax, height[r])
            res += rightMax - height[r]
    return res
# Quick debug — run this cell while building
print(trap([0,1,0,2,1,0,1,3,2,1,2,1]))  # 6
print(trap([4,2,0,3,2,5]))             # 9
print(trap([3,0,3]))                     # 3
print(trap([1,2,3,4,5]))                 # 0
test_harness(trap)
        
        


[debug] height=[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
6
[debug] height=[4, 2, 0, 3, 2, 5]
9
[debug] height=[3, 0, 3]
3
[debug] height=[1, 2, 3, 4, 5]
0
[debug] height=[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
[debug] height=[4, 2, 0, 3, 2, 5]
[debug] height=[3, 0, 3]
[debug] height=[1]
[debug] height=[1, 2]
[debug] height=[0, 0, 0]
[debug] height=[5, 4, 3, 2, 1]
[debug] height=[1, 2, 3, 4, 5]
[debug] height=[2, 0, 2]

Result: 9/9 passed
All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(trap)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (nested) | O(n²) | O(1) | For each bar, scan left+right |
| Prefix/suffix arrays | O(n) | O(n) | Pre-compute max from each side |
| **Two pointers** | **O(n)** | **O(1)** | **Single pass, optimal** |
| Stack-based | O(n) | O(n) | Horizontal layer counting |

## Real World Connection

In AWS data pipelines, this pattern mirrors **backpressure
analysis**: how much data can queue up between two processing
stages when throughput is constrained on one side.
At Citi, risk systems compute Value-at-Risk (VaR) windows
where exposure is bounded by the minimum of two worst-case
scenarios — exactly the min(left_max, right_max) logic.
Data engineers use similar two-pointer logic when aligning
time-series data from two sources with different frequencies,
always advancing the "slower" side.
The pattern teaches that you can often replace O(n) extra
memory with a pair of running state variables when the
problem has left-right symmetry.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra